# RSNA Knee Abnormality Detection · 7th place inference, eight models

This notebook reproduces my final submission 2: the four models of [final submission 1](https://www.kaggle.com/code/songqizhou/rsna-knee-7th-place-inference) plus a twin of each, **private LB 0.957, public LB 0.970**. Each twin uses its model's recipe with another seed and is trained on the training set + the 58 gold studies (with my labels, not the host labels).

**No external training data.** The models were trained only on the competition images, with labels read from the competition reports and images by an LLM, starting from ImageNet-pretrained backbones. Weights: the Kaggle Model [RSNA Knee 7th Place](https://www.kaggle.com/models/songqizhou/rsna-knee-7th-place/PyTorch/default) and the same files as the dataset [RSNA Knee 7th Place Models](https://www.kaggle.com/datasets/songqizhou/rsna-knee-7th-place-models), both attached. Training code: [GitHub](https://github.com/SongqiZhou/rsna-knee-abnormality-detection-7th-place).

| Model | Backbone (timm) | Attention temperature | Public | Private |
|---|---|---|---|---|
| EfficientNetV2-S, fixed window sampler | `tf_efficientnetv2_s.in21k_ft_in1k` | 1.75 | 0.968 | 0.954 |
| EfficientNetV2-S, progressive window sampler | `tf_efficientnetv2_s.in21k_ft_in1k` | 1.75 | 0.969 | 0.954 |
| ConvNeXt-S | `convnext_small.in12k_ft_in1k` | 1.25 | 0.969 | **0.958** |
| ConvNeXt-T | `convnext_tiny.in12k_ft_in1k` | 1.25 | 0.969 | 0.957 |
| **Rank mean of the four** (final submission 1) | | | **0.970** | **0.957** |
| **Rank mean of the eight** (this notebook) | | | **0.970** | **0.957** |

1. **Preprocessing.** DICOM → up to 6 series (sagittal, coronal and axial, each with and without fat suppression), cropped to 140 mm at 384 px, right knees mirrored, at most 48 slices per series.
2. **Windows.** Every slice is the centre of a 3-slice window. 13 fixed zoomed views (menisci, patellofemoral joint, compartments) reuse the windows of their parent series.
3. **Network.** A shared 2D CNN turns each window into 12×12 tokens with a series embedding and a 3D position in a common knee frame. A label-query decoder and a gated attention-MIL head read every token of the study.
4. **Ensemble.** Per label, the mean of the models' percentile ranks.

On the hidden test set the notebook runs in about 5 h on 2×T4 (0.6 h per model), within the 9 h limit.

## 1. Preprocessing
DICOM → one uint8 volume per series slot.

In [ ]:
%%writefile preprocess.py
"""DICOM -> one uint8 volume per series slot, the input format all four models were trained on.

- Series slot from the host CSV: {SAG, COR, AX} x {FS, N} (FS = Fat_Suppression == 1). If a study has two series
  for one slot, the one with more slices is kept.
- Slices in geometric order, at most 48 per series (even subsampling).
- In-plane orientation is canonicalised from ImageOrientationPatient, and right knees are mirrored into the
  left-knee frame (coronal / axial: flip columns; sagittal: reverse the slice order), so medial and lateral
  always sit on the same side.
- Fixed 140 mm crop around the image centre (smaller fields of view are padded, never stretched), resized to 384 px.
- Per-series 1-99 percentile window -> uint8.
"""
import os, glob
import numpy as np
import pydicom
import cv2

CROP_MM = 140.0
OUT_PX = 384
MAX_SLICES = 48
PLANE_STD = {  # canonical (row, col) directions in LPS patient space
    "SAG": (np.array([0, 1, 0.]), np.array([0, 0, -1.])),
    "COR": (np.array([1, 0, 0.]), np.array([0, 0, -1.])),
    "AX": (np.array([1, 0, 0.]), np.array([0, 1, 0.])),
}
PLANE_AXIS = {"SAG": 0, "COR": 1, "AX": 2}
CSV_PLANE = {"Sagittal": "SAG", "Coronal": "COR", "Axial": "AX"}


def _f(x, d=0.0):
    try:
        return float(x)
    except Exception:
        return d


def series_header(files, plane_hint=None):
    """Headers of one series -> its files sorted along the plane axis + geometry, or None."""
    hs = []
    for f in files:
        try:
            hs.append((f, pydicom.dcmread(f, stop_before_pixels=True)))
        except Exception:
            continue
    if not hs:
        return None
    d0 = hs[0][1]
    iop = getattr(d0, "ImageOrientationPatient", None)
    if iop is None:
        return None
    row, col = np.array(iop[:3], float), np.array(iop[3:], float)
    plane = plane_hint or ["SAG", "COR", "AX"][int(np.argmax(np.abs(np.cross(row, col))))]
    ax = PLANE_AXIS[plane]

    def pos(h):
        ipp = getattr(h, "ImagePositionPatient", None)
        if ipp is None:
            return _f(getattr(h, "InstanceNumber", 0))
        return float(np.array(ipp, float)[ax])   # ascending patient coordinate along the plane axis
    hs.sort(key=lambda t: pos(t[1]))
    ps = getattr(d0, "PixelSpacing", [0.5, 0.5])
    cxs = []   # x of the image centre (LPS, +x = patient left), the laterality fallback
    for _, h in hs[:: max(1, len(hs) // 8)]:
        ipp, p = getattr(h, "ImagePositionPatient", None), getattr(h, "PixelSpacing", None)
        if ipp is None or p is None:
            continue
        r_, c_ = int(getattr(h, "Rows", 0)), int(getattr(h, "Columns", 0))
        cxs.append((np.array(ipp, float) + row * _f(p[1]) * c_ / 2 + col * _f(p[0]) * r_ / 2)[0])
    lat = str(getattr(d0, "Laterality", "") or getattr(d0, "ImageLaterality", "") or "").upper()
    return {"files": [f for f, _ in hs], "plane": plane, "row": row, "col": col, "n": len(hs),
            "spacing": (_f(ps[0], 0.5), _f(ps[1], 0.5)), "lat_tag": lat if lat in ("L", "R") else "",
            "cx": float(np.median(cxs)) if cxs else 0.0, "thick": _f(getattr(d0, "SliceThickness", 0))}


def _pixels(f):
    d = pydicom.dcmread(f)
    px = d.pixel_array.astype(np.float32)
    if px.ndim == 3:
        px = px[px.shape[0] // 2]
    px = px * _f(getattr(d, "RescaleSlope", 1), 1) + _f(getattr(d, "RescaleIntercept", 0))
    if str(getattr(d, "PhotometricInterpretation", "")) == "MONOCHROME1":
        px = px.max() - px
    return px


def _crop_pad(px, sp, out_px):
    """Centre crop to CROP_MM (padded with the minimum if the field of view is smaller), resized to out_px."""
    sy, sx = sp
    ch, cw = int(round(CROP_MM / sy)), int(round(CROP_MM / sx))
    h, w = px.shape
    canvas = np.full((max(h, ch), max(w, cw)), px.min(), np.float32)
    oy, ox = (canvas.shape[0] - h) // 2, (canvas.shape[1] - w) // 2
    canvas[oy:oy + h, ox:ox + w] = px
    H, W = canvas.shape
    y0, x0 = (H - ch) // 2, (W - cw) // 2
    c = canvas[y0:y0 + ch, x0:x0 + cw]
    return cv2.resize(c, (out_px, out_px), interpolation=cv2.INTER_AREA)


def process_series(si, lat, out_px=OUT_PX, max_slices=MAX_SLICES):
    n = si["n"]
    idx = np.arange(n) if n <= max_slices else np.unique(np.linspace(0, n - 1, max_slices).round().astype(int))
    plane = si["plane"]
    r_std, c_std = PLANE_STD[plane]
    flip_c = float(np.dot(si["row"], r_std)) < 0
    flip_r = float(np.dot(si["col"], c_std)) < 0
    imgs = []
    for i in idx:
        try:
            imgs.append(_pixels(si["files"][i]))
        except Exception:
            imgs.append(None)
    good = [k for k, im in enumerate(imgs) if im is not None]
    if not good:
        return None
    for k in range(len(imgs)):   # an unreadable slice is replaced by its nearest readable neighbour
        if imgs[k] is None:
            imgs[k] = imgs[min(good, key=lambda j: abs(j - k))]
    out = []
    for im in imgs:
        if flip_c:
            im = im[:, ::-1]
        if flip_r:
            im = im[::-1, :]
        if lat == "R" and plane in ("COR", "AX"):
            im = im[:, ::-1]
        out.append(_crop_pad(np.ascontiguousarray(im), si["spacing"], out_px))
    vol = np.stack(out)
    if lat == "R" and plane == "SAG":
        vol = vol[::-1]
    sub = vol[:, ::4, ::4]
    p1, p99 = np.percentile(sub, [1, 99])
    vol = np.clip((vol - p1) / max(p99 - p1, 1e-3), 0, 1)
    return (vol * 255 + 0.5).astype(np.uint8)


def choose_slots(rows):
    """rows [(plane, fat_suppression, header)] -> {slot: header}; the series with more slices wins a slot."""
    chosen = {}
    for plane, fs, si in rows:
        if si is None:
            continue
        slot = f"{plane}_{'FS' if int(fs) == 1 else 'N'}"
        if slot not in chosen or si["n"] > chosen[slot]["n"]:
            chosen[slot] = si
    return chosen


def study_laterality(sis):
    """DICOM laterality tag if any series has one, else the side of the image centre, else left."""
    tags = [s["lat_tag"] for s in sis if s["lat_tag"]]
    if tags:
        return max(set(tags), key=tags.count)
    cx = float(np.median([s["cx"] for s in sis]))
    if cx > 20:
        return "L"
    if cx < -20:
        return "R"
    return "L"


def load_study(study_dir, series_df, out_px=OUT_PX):
    """One test study (study_dir/<series>/<sop>.dcm, its rows of test_series.csv) ->
    {slot: (uint8 volume (n, out_px, out_px), number of slices in the series, slice thickness in mm)}."""
    rows = []
    for r in series_df.itertuples():
        files = sorted(glob.glob(os.path.join(study_dir, str(r.SeriesInstanceUID), "*.dcm")))
        if len(files) < 3:
            continue
        si = series_header(files, CSV_PLANE.get(r.Anatomical_Plane))
        rows.append((CSV_PLANE.get(r.Anatomical_Plane, si["plane"] if si else "AX"), int(r.Fat_Suppression), si))
    sis = [si for _, _, si in rows if si is not None]
    if not sis:
        return {}
    lat = study_laterality(sis)
    out = {}
    for slot, si in choose_slots(rows).items():
        v = process_series(si, lat, out_px)
        if v is not None:
            out[slot] = (v, si["n"], si["thick"])
    return out

## 2. Network
The 2.5D label-query network shared by all models.

In [ ]:
%%writefile model.py
"""2.5D label-query network.

A study is a bag of 2.5D windows (3 neighbouring slices) from up to 6 series plus 13 zoomed views. One shared timm
backbone turns every window into a 12 x 12 grid of tokens. Each token gets an embedding of its view and of its 3D
position in a common knee frame (X lateral, Y posterior, Z inferior; mm / 70), so all planes and crops line up.
Two readouts, whose logits are averaged:

- label queries: 2 learnable queries per label; 2 pre-LN decoder layers (query self-attention -> cosine
  cross-attention over every token of every window -> FFN). A label's logit is a linear map of the last layer's
  cross-attention output.
- gated attention-MIL (Ilse et al. 2018) over window vectors (mean of the window's tokens + position).

At inference the attention logits of both readouts are multiplied by a temperature (> 1 = sharper), since a dense
test bag has many more windows than a training bag.
"""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

N_SERIES = 6                        # SAG_FS, SAG_N, COR_FS, COR_N, AX_FS, AX_N
PLANE_OF_SERIES = [0, 0, 1, 1, 2, 2]  # 0 sagittal, 1 coronal, 2 axial
HALF_FOV_MM = 70.0                  # the 140 mm crop


def knee_coords(u, v, w, plane):
    """In-plane (u = column, v = row) and through-plane (w = slice) offsets in mm -> (..., 3) knee frame / 70.
    Columns run posterior (sagittal) / lateral (coronal, axial); rows run inferior (sagittal, coronal) / posterior
    (axial); slices run lateral (sagittal) / posterior (coronal) / superior (axial)."""
    X = torch.where(plane == 0, w, u)
    Y = torch.where(plane == 0, u, torch.where(plane == 1, w, v))
    Z = torch.where(plane == 2, -w, v)
    return torch.stack([X, Y, Z], -1) / HALF_FOV_MM


class FourierPE(nn.Module):
    def __init__(self, D, n_freq=8):
        super().__init__()
        self.register_buffer("freq", (2.0 ** torch.arange(n_freq)) * math.pi / 2, persistent=False)
        self.mlp = nn.Sequential(nn.Linear(6 * n_freq, D), nn.GELU(), nn.Linear(D, D))

    def forward(self, c):                     # c (..., 3)
        a = c[..., None] * self.freq          # (..., 3, n_freq)
        return self.mlp(torch.cat([a.sin(), a.cos()], -1).flatten(-2))


class DecoderLayer(nn.Module):
    def __init__(self, D, heads, drop=0.1, ffn_mult=4):
        super().__init__()
        self.h = heads
        self.att_temp = 1.0
        self.log_s = nn.Parameter(torch.full((1, heads, 1, 1), math.log(5.0)))   # cosine-attention scale
        self.n1, self.n2, self.n3, self.nk = nn.LayerNorm(D), nn.LayerNorm(D), nn.LayerNorm(D), nn.LayerNorm(D)
        self.sa = nn.MultiheadAttention(D, heads, dropout=0.0, batch_first=True)
        self.q, self.k, self.v, self.o = (nn.Linear(D, D) for _ in range(4))
        self.ffn = nn.Sequential(nn.Linear(D, ffn_mult * D), nn.GELU(), nn.Dropout(drop), nn.Linear(ffn_mult * D, D))
        self.drop = nn.Dropout(drop)

    def forward(self, q, tok, pos, mask):
        """q (B, Q, D); tok / pos (B, T, D); mask (B, T) True = real token.
        Returns the updated queries and this layer's cross-attention output r (B, Q, D)."""
        x = self.n1(q)
        q = q + self.drop(self.sa(x, x, x, need_weights=False)[0])
        B, Q, D = q.shape
        T, h, dh = tok.shape[1], self.h, D // self.h
        t = self.nk(tok)
        qq = self.q(self.n2(q)).view(B, Q, h, dh).transpose(1, 2)
        kk = self.k(t + pos).view(B, T, h, dh).transpose(1, 2)
        vv = self.v(t).view(B, T, h, dh).transpose(1, 2)
        sc = (self.log_s.exp().clamp(max=20.0) * self.att_temp).to(qq.dtype)
        o = F.scaled_dot_product_attention(F.normalize(qq, dim=-1) * sc, F.normalize(kk, dim=-1), vv,
                                           attn_mask=mask[:, None, None, :], scale=1.0)
        r = self.o(o.transpose(1, 2).reshape(B, Q, D))
        q = q + self.drop(r)
        return q + self.drop(self.ffn(self.n3(q))), r


def _pad_rows(x, m):
    """Pad a (n, ...) batch to m rows by repeating its own rows."""
    n = len(x)
    if m <= n:
        return x
    return torch.cat([x, x[torch.arange(m - n, device=x.device) % n]])


class KQNet(nn.Module):
    def __init__(self, arch, roi_planes, D=384, n_lab=12, q_per=2, dec_layers=2, heads=8, drop=0.1):
        super().__init__()
        self.att_temp = 1.0
        self.bb = timm.create_model(arch, pretrained=False, num_classes=0, in_chans=3)
        self.n_lab, self.q_per = n_lab, q_per
        self.p4 = nn.Conv2d(self.bb.feature_info[-1]["num_chs"], D, 1)   # stride-32 features -> tokens
        self.tnorm = nn.LayerNorm(D)
        # view slots: the 6 series, then the zoomed views (crops of a parent series, same plane)
        self.register_buffer("plane_of_slot", torch.tensor(PLANE_OF_SERIES + list(roi_planes)), persistent=False)
        self.slot_emb = nn.Embedding(N_SERIES + len(roi_planes), D)
        self.pe = FourierPE(D)
        self.queries = nn.Parameter(torch.randn(n_lab * q_per, D) * 0.02)
        self.dec = nn.ModuleList([DecoderLayer(D, heads, drop) for _ in range(dec_layers)])
        self.qnorm = nn.LayerNorm(D)
        self.qW = nn.Parameter(torch.randn(n_lab, D) * 0.02)
        self.qb = nn.Parameter(torch.zeros(n_lab))
        self.wnorm = nn.LayerNorm(D)
        self.att_v, self.att_u = nn.Linear(D, 256), nn.Linear(D, 256)
        self.att_w = nn.Linear(256, n_lab)
        self.mW = nn.Parameter(torch.randn(n_lab, D) * 0.02)
        self.mb = nn.Parameter(torch.zeros(n_lab))
        self.bb_chunk = 32   # the backbone runs on fixed chunks of this many windows

    def set_att_temp(self, t):
        for m in self.modules():
            if hasattr(m, "att_temp"):
                m.att_temp = float(t)

    def _features(self, x):
        """Backbone in fixed-size chunks (a few input shapes instead of one per study); the last chunk is padded with
        repeated windows whose outputs are dropped."""
        c, outs = self.bb_chunk, []
        for i in range(0, len(x), c):
            xi = x[i:i + c]
            outs.append(self.bb.forward_intermediates(_pad_rows(xi, c), indices=1, intermediates_only=True)[-1][:len(xi)])
        return torch.cat(outs)

    def forward(self, x, wmeta, mask, theta):
        """x (B, W, 3, H, W) normalised windows; wmeta (B, W, 2) [view slot, slice offset mm]; mask (B, W) bool;
        theta (B, W, 2, 3) the affine that produced each window from its cached slice, used to place every token at
        its anatomical position."""
        B, Wn = mask.shape
        t = self.p4(self._features(x[mask]))
        n, D, gh, gw = t.shape
        t = self.tnorm(t.flatten(2).transpose(1, 2))                       # (n, G, D)
        slot = wmeta[..., 0].long()[mask]                                  # (n,)
        plane = self.plane_of_slot[slot]
        emb = self.slot_emb(slot)[:, None]                                 # (n, 1, D)
        ys = (torch.arange(gh, device=x.device) + 0.5) / gh * 2 - 1
        xs = (torch.arange(gw, device=x.device) + 0.5) / gw * 2 - 1
        gy, gx = torch.meshgrid(ys, xs, indexing="ij")
        g = torch.stack([gx.flatten(), gy.flatten(), torch.ones_like(gx.flatten())], -1)   # (G, 3)
        src = torch.einsum("nij,gj->ngi", theta[mask].float(), g)          # (n, G, 2) position in the cached slice
        w = wmeta[..., 1][mask].float()[:, None].expand(-1, src.shape[1])
        c = knee_coords(src[..., 0] * HALF_FOV_MM, src[..., 1] * HALF_FOV_MM, w, plane[:, None].expand_as(w))
        pos = self.pe(c).to(t.dtype) + emb.to(t.dtype)                     # (n, G, D)
        G = t.shape[1]
        tok = t.new_zeros(B, Wn, G, D); tok[mask] = t
        ps = t.new_zeros(B, Wn, G, D); ps[mask] = pos
        # label queries over every token of every window
        tm = mask[:, :, None].expand(B, Wn, G).reshape(B, Wn * G)
        q = self.queries[None].expand(B, -1, -1).to(t.dtype)
        tk, pk = tok.reshape(B, Wn * G, D), ps.reshape(B, Wn * G, D)
        for layer in self.dec:
            q, r = layer(q, tk, pk, tm)
        r = self.qnorm(r.view(B, self.n_lab, self.q_per, D).mean(2)).float()
        logit_q = (r * self.qW).sum(-1) + self.qb
        # gated attention-MIL over window vectors
        logit_m = self._mil(tok.mean(2) + ps.mean(2), mask)
        return (logit_q + logit_m) / 2

    def _mil(self, hw, mask):
        """Gated attention-MIL over window vectors hw (B, W, D) -> (B, n_lab) logits."""
        h = self.wnorm(hw)
        a = self.att_w(torch.tanh(self.att_v(h)) * torch.sigmoid(self.att_u(h))).float() * self.att_temp
        a = torch.softmax(a.masked_fill(~mask[..., None], -1e4), 1)        # (B, W, n_lab)
        pooled = torch.einsum("bwn,bwd->bnd", a, h.float())
        return (pooled * self.mW).sum(-1) + self.mb

## 3. Inference and rank mean
CPU processes prepare the studies, one thread per GPU runs the models, and the per-label rank mean is written to `submission.csv`.

In [ ]:
%%writefile infer.py
"""Inference: DICOM -> 2.5D windows -> four (or eight) models -> per-label rank mean -> submission.csv.

- Preprocessing (preprocess.py) runs in a pool of CPU processes, a few studies ahead of the GPUs.
- One worker thread per GPU, each with its own copy of every model; fp16 autocast.
- Windows: every slice of every series is a window centre (dense stride 1); each zoomed view uses the windows of
  its parent series and crops them on the GPU.
- Ensemble: per label, the mean over the models of their percentile ranks across the test studies.
- Paths: environment variables COMP_DIR / MODEL_DIR / OUT, else SETTINGS.json at the repository root, else the
  Kaggle input folders (the notebook).
usage: [ENSEMBLE=4|8] python inference/infer.py
"""
import os, sys, glob, json, time, queue, threading
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
import numpy as np, pandas as pd
import torch
import torch.nn.functional as F

sys.dont_write_bytecode = True   # keep the output folder clean
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from preprocess import load_study  # noqa: E402
from model import KQNet  # noqa: E402

LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA",
          "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
SERIES = ["SAG_FS", "SAG_N", "COR_FS", "COR_N", "AX_FS", "AX_N"]
PLANE = {"SAG": 0, "COR": 1, "AX": 2}
# 13 zoomed views: (parent series, y0, y1, x0, x1) as fractions of the 140 mm slice. They cover the menisci, the
# patellofemoral joint and the medial / lateral compartments, each enlarged to 384 px.
ROIS = [("AX_FS", 0.0, 0.6, 0.2, 0.8), ("SAG_FS", 0.1, 0.7, 0.0, 0.6), ("COR_FS", 0.2, 0.8, 0.4, 1.0),
        ("COR_FS", 0.2, 0.8, 0.0, 0.6), ("SAG_FS", 0.38, 0.83, 0.1, 0.55), ("SAG_FS", 0.38, 0.83, 0.45, 0.9),
        ("SAG_N", 0.38, 0.83, 0.45, 0.9), ("COR_FS", 0.3, 0.75, 0.08, 0.53), ("COR_FS", 0.3, 0.75, 0.47, 0.92),
        ("AX_FS", 0.02, 0.5, 0.2, 0.8), ("AX_N", 0.02, 0.5, 0.2, 0.8), ("SAG_FS", 0.12, 0.62, 0.08, 0.58),
        ("SAG_N", 0.12, 0.62, 0.08, 0.58)]
# The two selected submissions, models in the order their percentile ranks are averaged. Final submission 1: the four
# models. Final submission 2: the same four + a twin of each (another seed, trained on the training set + the 58 gold
# studies). Both scored 0.970 public / 0.957 private. ENSEMBLE=4 (default) or ENSEMBLE=8 picks one.
ENSEMBLES = {"4": ["effnetv2s_fixed.pt", "effnetv2s_progressive.pt", "convnext_small.pt", "convnext_tiny.pt"],
             "8": ["effnetv2s_fixed.pt", "effnetv2s_fixed_twin.pt", "effnetv2s_progressive.pt",
                   "effnetv2s_progressive_twin.pt", "convnext_small.pt", "convnext_small_twin.pt", "convnext_tiny.pt",
                   "convnext_tiny_twin.pt"]}
MODELS = ENSEMBLES[os.environ.get("ENSEMBLE", "4")]
SPAN = (0.04, 0.96)   # window centres cover the central 92% of each series
RES = 384
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 1, 3, 1, 1)
TIME_BUDGET_S = 8 * 3600   # past this, stop and submit the studies done so far (the limit is 9 h)
ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))


def setting(key):
    """A path from SETTINGS.json at the repository root (relative paths are relative to that root), or None."""
    f = os.path.join(ROOT, "SETTINGS.json")
    if not os.path.exists(f):
        return None
    v = json.load(open(f)).get(key)
    return v if v is None or os.path.isabs(v) else os.path.join(ROOT, v)


def find_dir(fname):
    for depth in range(1, 6):
        hits = glob.glob("/kaggle/input/" + "*/" * depth + fname)
        if hits:
            return os.path.dirname(hits[0])
    raise FileNotFoundError(fname)


def study_windows(vols):
    """{slot: (volume (n, 384, 384) uint8, slices in the series, thickness mm)} -> windows (W, 3, 384, 384) uint8
    and meta (W, 2) float32 = [view slot, slice offset from the series centre in mm]."""
    wins, meta = [], []
    for slot in SERIES:
        if slot not in vols:
            continue
        v, n_orig, thick = vols[slot]
        if v.ndim != 3 or v.shape[1:] != (RES, RES) or len(v) < 1:
            continue
        n = len(v)
        lo, hi = max(1, int(round(SPAN[0] * (n - 1)))), min(n - 2, int(round(SPAN[1] * (n - 1))))
        cs = np.arange(lo, hi + 1) if hi >= lo else np.array([n // 2])
        idx = np.clip(cs[:, None] + np.arange(-1, 2)[None], 0, n - 1)    # 3 neighbouring slices per window
        step_mm = (thick if thick > 0 else 3.5) * max(n_orig, n) / n       # spacing of the kept slices
        off = ((cs - (n - 1) / 2) * step_mm).astype(np.float32)
        for sid in [SERIES.index(slot)] + [6 + i for i, r in enumerate(ROIS) if r[0] == slot]:
            wins.append(v[idx])
            meta.append(np.stack([np.full(len(cs), sid, np.float32), off], 1))
    if not wins:
        return None
    return np.concatenate(wins), np.concatenate(meta)


def view_affines(device):
    """(6 + 13, 2, 3) affine per view slot, in normalised [-1, 1] coordinates of the cached slice: a series shows the
    whole slice; zoomed view i shows its crop, enlarged to the model resolution."""
    th = torch.zeros(6 + len(ROIS), 2, 3)
    th[:6, 0, 0] = th[:6, 1, 1] = 1.0
    crop = torch.tensor([[x0 + x1 - 1, y0 + y1 - 1, x1 - x0, y1 - y0] for _, y0, y1, x0, x1 in ROIS], dtype=torch.float32)
    th[6:, 0, 0], th[6:, 1, 1] = crop[:, 2], crop[:, 3]                    # half-size of the crop
    th[6:, 0, 2], th[6:, 1, 2] = crop[:, 0], crop[:, 1]                    # centre of the crop
    return th.to(device)


def gpu_views(X, Mt, M, affines):
    """Windows X (B, W, 3, 384, 384) uint8 on the GPU -> model input (B, W, 3, RES, RES) in [0, 1] and theta
    (B, W, 2, 3), the affine of each window's view, which the model also uses to place its tokens."""
    B, W = M.shape
    theta = affines[Mt[..., 0].long().clamp(0, len(affines) - 1)]          # (B, W, 2, 3)
    xf = X[M].float() / 255.0
    grid = F.affine_grid(theta[M], (len(xf), 3, RES, RES), align_corners=False)
    out = F.grid_sample(xf, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    full = torch.zeros(B, W, 3, RES, RES, device=X.device, dtype=out.dtype)
    full[M] = out
    return full, theta


def load_models(model_dir, devs):
    """-> one list per model with a copy on every device."""
    roi_planes = [PLANE[r[0].split("_")[0]] for r in ROIS]
    models = []
    for f in MODELS:
        ck = torch.load(os.path.join(model_dir, f), map_location="cpu", weights_only=True)
        cfg = ck["config"]
        copies = []
        for dev in devs:
            m = KQNet(cfg["arch"], roi_planes, D=cfg["dim"])
            m.load_state_dict(ck["model"])
            m.set_att_temp(cfg["att_temp"])
            m.eval().to(dev)
            if dev != "cpu":
                m.bb.to(memory_format=torch.channels_last)
            copies.append(m)
        models.append(copies)
        print(f"{f}: {cfg['arch']}, attention temperature {cfg['att_temp']}", flush=True)
    return models


@torch.no_grad()
def predict_study(models, di, affines, wins, meta):
    """-> (n_models, 12) probabilities for one study, on device index di."""
    dev = affines.device
    X = torch.from_numpy(wins)[None].to(dev, non_blocking=True)
    Mt = torch.from_numpy(meta)[None].to(dev)
    M = torch.ones(1, len(wins), dtype=torch.bool, device=dev)
    xv, theta = gpu_views(X, Mt, M, affines)
    xv = (xv - MEAN.to(dev)) / STD.to(dev)
    out = []
    for copies in models:
        with torch.autocast("cuda", dtype=torch.float16, enabled=dev.type != "cpu"):
            o = copies[di](xv, Mt, M, theta)
        out.append(torch.sigmoid(o.float()).cpu().numpy()[0])
    return np.stack(out)


def rank_mean(P):
    """P (N, n_models, 12) probabilities (nan = study failed) -> (N, 12): per label, the mean over models of their
    percentile ranks across the test studies; a failed study gets the middle rank."""
    n = len(P)
    out = np.full((n, 12), 0.5)
    for c in range(12):
        ranks = []
        for mi in range(P.shape[1]):
            col = P[:, mi, c]
            ok = ~np.isnan(col)
            if ok.sum() < 2:
                continue
            r = np.full(n, 0.5)
            r[ok] = pd.Series(col[ok]).rank(pct=True).values
            ranks.append(r)
        if ranks:
            out[:, c] = np.mean(ranks, axis=0)
    return out


def write_submission(path, uids, P):
    sub = pd.DataFrame(rank_mean(P), columns=LABELS)
    sub.insert(0, "StudyInstanceUID", uids)
    sub.to_csv(path, index=False)


def _load(study_dir, series_df):
    try:
        return load_study(study_dir, series_df)
    except Exception as e:
        print("preprocessing failed", os.path.basename(study_dir), repr(e), flush=True)
        return {}


def main():
    t0 = time.time()
    comp = os.environ.get("COMP_DIR") or setting("RAW_DATA_DIR") or find_dir("test_series.csv")
    model_dir = os.environ.get("MODEL_DIR") or setting("MODEL_DIR") or find_dir(MODELS[0])
    out_csv = os.environ.get("OUT") or (os.path.join(setting("SUBMISSION_DIR"), "submission.csv") if setting("SUBMISSION_DIR")
                                        else "submission.csv")
    os.makedirs(os.path.dirname(os.path.abspath(out_csv)), exist_ok=True)
    uids = pd.read_csv(f"{comp}/test.csv").StudyInstanceUID.astype(str).tolist()
    ser = pd.read_csv(f"{comp}/test_series.csv")
    ser["StudyInstanceUID"] = ser.StudyInstanceUID.astype(str)
    N = len(uids)
    P = np.full((N, len(MODELS), 12), np.nan, np.float32)
    write_submission(out_csv, uids, P)   # a valid file from the start
    print(f"{N} test studies", flush=True)

    ncpu = max(1, min(4, os.cpu_count() or 4))
    pool = ProcessPoolExecutor(max_workers=ncpu, mp_context=mp.get_context("fork"))   # fork before CUDA starts
    list(pool.map(abs, range(ncpu)))
    devs = [f"cuda:{i}" for i in range(torch.cuda.device_count())] or ["cpu"]
    models = load_models(model_dir, devs)
    affines = [view_affines(torch.device(d)) for d in devs]
    print(f"{len(MODELS)} models x {len(devs)} devices ({time.time() - t0:.0f}s)", flush=True)

    work_q = queue.Queue(maxsize=8)

    def gpu_worker(di):
        while (item := work_q.get()) is not None:
            i, vols = item
            try:
                w = study_windows(vols)
                if w is not None:
                    P[i] = predict_study(models, di, affines[di], *w)
            except Exception as e:   # an uncaught error would stop this thread and block the queue
                print("study failed", uids[i], repr(e), flush=True)
                torch.cuda.empty_cache()

    threads = [threading.Thread(target=gpu_worker, args=(di,), daemon=True) for di in range(len(devs))]
    for t in threads:
        t.start()
    t_loop = time.time()
    with pool:
        ahead = 16
        futs = {i: pool.submit(_load, f"{comp}/test_series/{uids[i]}", ser[ser.StudyInstanceUID == uids[i]])
                for i in range(min(ahead, N))}
        nxt = len(futs)
        for i in range(N):
            if time.time() - t0 > TIME_BUDGET_S:
                print("time budget reached at study", i, flush=True)
                break
            vols = futs.pop(i).result()
            if nxt < N:
                futs[nxt] = pool.submit(_load, f"{comp}/test_series/{uids[nxt]}", ser[ser.StudyInstanceUID == uids[nxt]])
                nxt += 1
            if vols:
                work_q.put((i, vols))
            if i % 50 == 0:
                print(f"{i}/{N} {time.time() - t0:.0f}s ({(time.time() - t_loop) / (i + 1):.2f}s/study)", flush=True)
            if i % 200 == 0 and i:
                write_submission(out_csv, uids, P)
    for _ in threads:
        work_q.put(None)
    for t in threads:
        t.join()
    write_submission(out_csv, uids, P)
    print(f"done in {time.time() - t0:.0f}s | studies with predictions {int((~np.isnan(P[:, 0, 0])).sum())}/{N}", flush=True)


if __name__ == "__main__":
    main()

## 4. Run

In [ ]:
import torch, timm
print("torch", torch.__version__, "| timm", timm.__version__, "| GPUs", torch.cuda.device_count())

In [ ]:
!ENSEMBLE=8 python infer.py

In [ ]:
import pandas as pd
pd.read_csv("submission.csv").head()